# DS4DS Exercise Sheet 03

In this exercise you implement your own numerical solver for the 2D heat equation, which describes the tempearture $T$ in a 2D domain over the time interval $t\in[t_0=0,t_e=0.2]$:

$$
\frac{\partial T}{\partial t} = \lambda \Delta T,
$$

where  
$\Delta T = \frac{\partial^2 T}{\partial s_1^2} + \frac{\partial^2 T}{\partial s_2^2}$ is the *Laplace operator* and $\lambda > 0$ is a material parameter.

The system can be modeled by a discrete approximation of the Laplace operator using the central difference method:

$$
\frac{\partial^2 T_{i,j}}{\partial s_1^2} \approx \frac{T_{i+1,j} - 2T_{i,j} + T_{i-1,j}}{\Delta s^2}, \qquad 
\frac{\partial^2 T_{i,j}}{\partial s_2^2} \approx \frac{T_{i,j+1} - 2T_{i,j} + T_{i,j-1}}{\Delta s^2}.
$$


As a setup, consider a rectangular domain $\Omega = (0,1)^2$ with boundary 
$$
\Gamma = [0,1] \times 0 ~\cup~ [0,1] \times 1 ~\cup~ 0 \times [0,1] ~\cup~ 1 \times [0,1]. 
$$ 
(i.e., bottom $\cup$ top $\cup$ left $\cup$ right.)

For discretization, use an equidistant grid with $n+2$ points in each direction (i.e., we have an $n+2 \times n+2$ grid in space). This means that $\Delta s = 1 / (n + 1)$. Of these $n+2$ grid points per dimension, the first and last are boundary nodes of Dirichlet type, meaning that they have a given value $T(s,t)= T_b(t)$ for $s\in\Gamma$. Consequently, the solution for the heat equation only has to be calculated for the interior of the domain. The solution we compute only concerns the $n^2$ interior grid nodes.

As this is more appropriate for numerical simulations and for later use of the resulting data, we will _vectorize_ (or _flatten_) the numerical solution into a vector $\hat{T}_i \in \mathbb{R}^{n^2}$ at every point $i$ in time. The indexing only addresses these interior points! This is visualized in the following sketch:

![](MatrixScetch.png)

Notes:

In 1D, heat flow at a specific point depends on the point to its left and right.In 2D, you must look at both the $x$ and $y$ directions. To find the second spatial derivative (the Laplace operator, $\Delta T$), you use the 5-point stencil.If you add the two central difference approximations from your first image together, you get:$$\Delta T_{i,j} \approx \frac{T_{i+1, j} + T_{i-1, j} + T_{i, j+1} + T_{i, j-1} - 4T_{i, j}}{\Delta s^2}$$In plain English: The heat change at the center node depends on the sum of its four neighbors (Left, Right, Top, Bottom) minus 4 times its own temperature, all scaled by the grid spacing $\Delta s^2$.

### Task 1: Create the heat equation matrix

First, consider homogeneous boundary conditions, i.e., $T(s,t)=0$ for $s\in\Gamma$. This means that we can discretize the system of interior points by a matrix-vector multiplication $\Delta T \approx A \hat{T}$ without including any boundary conditions.
The resulting estimated derivative is formed as
$$
\frac{\partial \hat{T}}{\partial t} = \lambda A \hat{T},
$$ 
where $\hat{T} \in \mathbb{R}^{N} = \mathbb{R}^{n^2}$, $A \in \mathbb{R}^{N \times N} = \mathbb{R}^{n^2 \times n^2}$.



Write a function to automatically compute the matrix $A$ for arbitrary grid sizes $n$ (per dimension).

Since the numerical solution (spatially discretized) $\hat{T}$ will have the form of a vector (and not a matrix), the grid nodes should be numbered according to the following convention (see also the Figure above):

* first row (i.e., $s_2 = \Delta s$) goes from $1$ to $n$,
* second row (i.e, $s_2 = 2\Delta s$) goes from $n + 1$ to $2n$,
* ...

This results in the form

$$
    \hat{T} = \begin{bmatrix} \hat{T}_0 & \hat{T}_1 & \dots & \hat{T}_{n-1} & \hat{T}_{n} & \hat{T}_{n+1} & \dots & \hat{T}_{n^2 - 1} & \hat{T}_{n^2} \end{bmatrix}^\mathrm{T}
$$



**Hint**: It is helpful to explicitly draw an example for, e.g., $n=3$ or $n=4$ on paper and compute the matrix by hand. What patterns do you see? How can you translate those patterns into rules for the matrix?

In [1]:
import numpy as np

def create_heat_equation_matrix(n):
    """
    Creates the 2D discrete Laplace operator matrix for an n x n interior grid.
    Includes the 1/ds^2 scaling factor.
    """
    N = n * n
    # Initialize a matrix of zeros
    A = np.zeros((N, N))
    
    # Calculate grid spacing delta s
    ds = 1.0 / (n + 1)
    
    # Iterate through every node in the flattened grid
    for i in range(N):
        
        # 1. Main Diagonal (The node itself)
        A[i, i] = -4.0
        
        # 2. Left Neighbor
        # If the index is perfectly divisible by n, it is on the left edge.
        if i % n != 0:          
            A[i, i - 1] = 1.0
            
        # 3. Right Neighbor
        # If the next index is perfectly divisible by n, it is on the right edge.
        if (i + 1) % n != 0:    
            A[i, i + 1] = 1.0
            
        # 4. Bottom Neighbor (Row below)
        # If the index is smaller than n, it is in the very first bottom row.
        if i >= n:              
            A[i, i - n] = 1.0
            
        # 5. Top Neighbor (Row above)
        # If the index is within the last n elements, it is in the very top row.
        if i < N - n:           
            A[i, i + n] = 1.0
            
    # Finally, apply the 1/ds^2 scaling factor to the entire matrix
    A = A * (1.0 / (ds ** 2))
    
    return A

# print(create_heat_equation_matrix(3))

### Task 2: Utilizing the explicit euler for time stepping 
Simulate the system (using the _Explicit Euler_ scheme for time stepping) with 

* $n=49$ (i.e., $\Delta s = 0.02$),
* $\Delta t=5 \cdot 10^{-5}$,
* $t_0 = 0$, 
* $t_e = 0.2$,
* initial state $T(s,0) = 1$ for $s\in\Omega$,
* $\lambda = 1$ ($\lambda$ should still be a parameter that can be changed).

Include a condition that identifies the time ```t_stat``` when a stationary state is almost reached ( $\| \hat{T}_i - \hat{T}_{i-1} \|_2 \leq 0.01$ ).
The solution function should return both the temperatures from $t_0$ until $t_e$ and the time satisfying the stationarity condition given above, i.e., ``` T, t_stat = heat_equation(n, λ, t_e, delta_t) ```.

**Hint 1**: You may use the `norm()` function from the `LinearAlgebra` package to compute $\|\cdot\|_2$.

**Hint 2**: The following two pictures visualize how the solution should look at t=0.05 and at t=0.1

t = 0.05                   |  t = 0.1
:-------------------------:|:-------------------------:
![](005.png)               | ![](01.png)


In [ ]:
def heat_equation(n, lam, t_e, delta_t):
    # Call the matrix creation function we built in the previous step
    A = create_heat_equation_matrix(n)

    # number of iterations
    n_iter = int(t_e / delta_t)

    # flag whether final state was found
    # when no termination occurs, return the -1
    t_stat = -1

    # create a matrix where T[:, 1] is the initial state, 
    # T[:, 2] is the state after the first iteration,
    # T[:, 3] is the state after the secound iteration, ... 
    T = np.zeros((n*n, n_iter + 1)) #for 2D grid, we have n*n nodes (flattening of the 2D grid), but for 1D, we have n nodes.

    # initial state T_0
    T_0 = np.ones(n * n) #first column of T is the initial state, which is a vector of ones (flattened 2D grid)
    T[:, 0] = T_0


    ### BEGIN SOLUTION
    
    # Loop from step 1 up to n_iter (inclusive)
    for k in range(1, n_iter + 1):
        
        # 1. Explicit Euler Time Step
        # Formula: T_new = T_old + dt * lambda * A * T_old
        # In Python, the '@' symbol performs fast matrix-vector multiplication
        T[:, k] = T[:, k-1] + delta_t * lam * (A @ T[:, k-1])
        
        # 2. Stationarity Check
        # Only check if we haven't already found the stationary time
        if t_stat == -1.0:
            
            # Calculate the L2 norm (Euclidean distance) between current and previous step
            change_norm = np.linalg.norm(T[:, k] - T[:, k-1])
            
            # If the change is small enough, record the physical time
            if change_norm <= 0.01:
                t_stat = k * delta_t
                
    ### END SOLUTION

    return T, t_stat

# --- Example Execution to test the function ---
# n = 49
# lam = 1.0
# t_e = 0.2
# delta_t = 5e-5
# T_history, t_stationary = heat_equation(n, lam, t_e, delta_t)
# print(f"Stationary state reached at t = {t_stationary:.4f} seconds")


Stationary state reached at t = 0.0703 seconds


### Task 3: Eigenvalue analysis

Examination of the eigenvalues of the matrix $A$ with the parameters of task 2. What can you say about the stability of the system? To this end, identify the eigenvalue ``` lambda_max ``` with the largest real part.

The system is:
- Stable                 -> stability = 1
- Asymptotically stable  -> stability = 2
- Unstable               -> stability = 3

In [3]:
n = 49

### BEGIN SOLUTION

# 1. Generate Matrix A
# (Assuming your create_heat_equation_matrix function is defined above)
A = create_heat_equation_matrix(n)

# 2. Calculate eigenvalues
# Because the discrete Laplacian matrix A is perfectly symmetric, 
# we can use eigvalsh() instead of eigvals() for much faster and more accurate computation.
eigenvalues = np.linalg.eigvalsh(A)

# 3. Identify the eigenvalue with the largest real part
lambda_max = np.max(np.real(eigenvalues))

# 4. Determine Stability
# We evaluate the continuous system dx/dt = A*x.
# We use a small tolerance (1e-10) to account for floating-point arithmetic errors.
if lambda_max < -1e-10:
    stability = 2  # Asymptotically stable (all real parts < 0)
elif lambda_max > 1e-10:
    stability = 3  # Unstable (at least one real part > 0)
else:
    stability = 1  # Stable / Marginally stable (largest real part is exactly 0)

### END SOLUTION

print(f"lambda_max: {lambda_max}")
print(f"stability:  {stability}")

lambda_max: -19.73271571734621
stability:  2


### Task 4: Analysis of various material constants 

To study the influence of the material constant $\lambda$, calculate the time until the system reaches the final state (using the same setup as in Task 2) for

* $\lambda = 0.5$
* $\lambda = 1.5$

Store the results in ```t_stat2``` and ```t_stat3```.

In [4]:
n = 49
t_e = 0.2
delta_t = 5 * 1e-5

### BEGIN SOLUTION

# We use the underscore '_' to ignore the first return value (the full temperature history matrix 'T') 
# because we only care about the second return value (the stationary time 't_stat').

# Calculate time to stationary state for lambda = 0.5
_, t_stat2 = heat_equation(n, 0.5, t_e, delta_t)

# Calculate time to stationary state for lambda = 1.5
_, t_stat3 = heat_equation(n, 1.5, t_e, delta_t)

### END SOLUTION

print(f"Stationary time for λ = 0.5: {t_stat2}")
print(f"Stationary time for λ = 1.5: {t_stat3}")

Stationary time for λ = 0.5: 0.0712
Stationary time for λ = 1.5: 0.06055
